In [32]:
from sklearn.datasets import load_diabetes
dibt = load_diabetes()

In [33]:
X = dibt.data
y = dibt.target

In [34]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [35]:
# diabetes 데이터셋의 경우 층화추출을 하면 에러가 나는데
# 1. 연속형 수치 변수를 층화추출을 시도
# 2. 클래스 불규형 문제
# K-fold 등의 우회적 방법으로 해결해야함
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42
)

scaler = StandardScaler()
X_train_zs = scaler.fit_transform(X_train)
X_test_zs = scaler.transform(X_test)

In [36]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
dibt_nn = MLPRegressor(hidden_layer_sizes=(64, 128, 64), max_iter=2000)
dibt_nn.fit(X_train_zs, y_train)
pred = dibt_nn.predict(X_test_zs)
print(mean_squared_error(y_test, pred))

7876.920035888647


# diabetes 데이터를 이용하여, knn, nb, dt, rf, ab, lr, ridge, lasso, mlp, svm 모델링하여 CV 로 MSE 로 성능을 비교하시오.
1. pipeline 사용
2. 제시된 모델 모두 사용
3. 최적의 파라미터와 성능 제시

In [37]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, GridSearchCV

In [38]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import BayesianRidge, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR

In [39]:
models = {
    "knn": (KNeighborsRegressor(), {"model__n_neighbors": [3, 5, 7, 9]}),
    "bayes": (BayesianRidge(), {}),
    "dt": (DecisionTreeRegressor(random_state=42), {"model__max_depth": [3, 5, 7, None]}),
    "rf": (RandomForestRegressor(random_state=42), {"model__n_estimators": [100, 200]}),
    "ab": (AdaBoostRegressor(random_state=42), {"model__n_estimators": [50, 100]}),
    "lr": (LinearRegression(), {}),
    "ridge": (Ridge(), {"model__alpha": [0.1, 1.0, 10.0]}),
    "lasso": (Lasso(max_iter=5000), {"model__alpha": [0.01, 0.1, 1.0]}),
    "mlp": (MLPRegressor(max_iter=5000, random_state=42), {"model__hidden_layer_sizes": [(64,), (64, 128, 64)]}),
    "svm": (SVR(), {"model__C": [0.1, 1.0, 10.0], "model__kernel": ["rbf", "linear"]}),
}

In [40]:
results = {}

for name, (estimator, param_grid) in models.items():
    pipe = Pipeline([
        ("sc", StandardScaler()),
        ("model", estimator),
    ])
    grid = GridSearchCV(
        pipe, param_grid,
        scoring="neg_mean_squared_error",
        cv=5,
    )
    grid.fit(X, y)

    best_mse = -grid.best_score_
    results[name] = (best_mse, grid.best_params_)
    print(f"{name:6s} | MSE={best_mse:9.2f} | best={grid.best_params_}")

knn    | MSE=  3344.79 | best={'model__n_neighbors': 9}
bayes  | MSE=  3001.79 | best={}
dt     | MSE=  3955.36 | best={'model__max_depth': 3}
rf     | MSE=  3346.63 | best={'model__n_estimators': 200}
ab     | MSE=  3393.02 | best={'model__n_estimators': 50}
lr     | MSE=  2993.08 | best={}
ridge  | MSE=  2993.02 | best={'model__alpha': 0.1}
lasso  | MSE=  2992.20 | best={'model__alpha': 0.1}


C:\dev\sesac\summary-260706\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5000) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\dev\sesac\summary-260706\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5000) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\dev\sesac\summary-260706\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5000) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\dev\sesac\summary-260706\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5000) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\dev\sesac\summary-260706\.ven

mlp    | MSE=  3178.14 | best={'model__hidden_layer_sizes': (64,)}
svm    | MSE=  3029.29 | best={'model__C': 1.0, 'model__kernel': 'linear'}


In [41]:
import numpy as np
print("=== MSE 낮은 순 (좋은 순) ===")
for name, (mse, params) in sorted(results.items(), key=lambda x: x[1][0]):
    print(f"{name:6s} | MSE={mse:9.2f} | {params} RMSE={np.sqrt(mse):9.2f}")

=== MSE 낮은 순 (좋은 순) ===
lasso  | MSE=  2992.20 | {'model__alpha': 0.1} RMSE=    54.70
ridge  | MSE=  2993.02 | {'model__alpha': 0.1} RMSE=    54.71
lr     | MSE=  2993.08 | {} RMSE=    54.71
bayes  | MSE=  3001.79 | {} RMSE=    54.79
svm    | MSE=  3029.29 | {'model__C': 1.0, 'model__kernel': 'linear'} RMSE=    55.04
mlp    | MSE=  3178.14 | {'model__hidden_layer_sizes': (64,)} RMSE=    56.37
knn    | MSE=  3344.79 | {'model__n_neighbors': 9} RMSE=    57.83
rf     | MSE=  3346.63 | {'model__n_estimators': 200} RMSE=    57.85
ab     | MSE=  3393.02 | {'model__n_estimators': 50} RMSE=    58.25
dt     | MSE=  3955.36 | {'model__max_depth': 3} RMSE=    62.89


In [42]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([("sc", StandardScaler()), ("model", Lasso(alpha=0.1, max_iter=5000))])
r2 = cross_val_score(pipe, X, y, scoring="r2", cv=5)
print(f"R²: {r2.mean():.3f}")   # 대략 0.48 근처

R²: 0.482


In [43]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR
from sklearn.linear_model import Ridge

In [44]:
# 스텝을 grid에서 통째로 교체할 것이므로, 여기 초기값은 자리표시자(placeholder)일 뿐
pipe = Pipeline([
    ("pre", StandardScaler()),
    ("model", SVR()),
])

In [45]:
# 리스트로 나누면 각 딕셔너리가 독립된 탐색 공간이 됨
# → linear에는 gamma를 조합하지 않아 중복 계산을 막음
param_grid = [
    # SVR + linear (gamma 불필요)
    {"pre": [StandardScaler(), MinMaxScaler()],
     "model": [SVR()],
     "model__kernel": ["linear"],
     "model__C": [1, 100]},

    # SVR + rbf (gamma 탐색)
    {"pre": [StandardScaler(), MinMaxScaler()],
     "model": [SVR()],
     "model__kernel": ["rbf"],
     "model__gamma": [0.01, 0.1, 1, 10],
     "model__C": [1, 100]},

    # Ridge (모델 자체를 교체해서 비교)
    {"pre": [StandardScaler(), MinMaxScaler()],
     "model": [Ridge()],
     "model__alpha": [0.1, 1.0, 10.0]},
]


In [46]:
grid = GridSearchCV(
    pipe, param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
)
grid.fit(X, y)

print("최적 조합:", grid.best_params_)
print(f"최적 MSE: {-grid.best_score_:.2f}")


최적 조합: {'model': SVR(), 'model__C': 100, 'model__gamma': 0.01, 'model__kernel': 'rbf', 'pre': StandardScaler()}
최적 MSE: 2938.69
